# 05 – Leakage-Controlled Validation (optional analysis)

**ENGE707 Assessment 3 – Facial Feature Classification Challenge (Kaggle)**

**Purpose.** The report shows that 143 of the 12,732 validation images (1.12%) have an exact or near duplicate in the
training subset, so validation accuracy could be slightly optimistic. This notebook measures how much that matters: it
re-scores the **saved** validation predictions twice, once on all validation images and once **without the flagged
images**, and reports errors and accuracy for both. Nothing is retrained and no images are read.

**Inputs.** `../docs/data_quality_check.csv` (per-image MD5 and difference hash, written by `01_eda.ipynb`),
`../src/pipeline.py`, and the saved validation probabilities `../models/val_probs.npy` (EfficientNetB0) and
`../models/val_probs_r50.npy` (ResNet50V2) from run 1.

**Outputs.** Printed results only; no files are written.

**How to run.** Run `01_eda.ipynb`, `03_model.ipynb` (and, if the run-1 files were restored, `03_model_rerun.ipynb`) first,
then run these cells from the `notebooks/` folder. It takes a minute or two on a CPU.

**Limits.** This only controls for *hash-detected* duplicates. It cannot detect different crops of the same photograph
(for example the left eye in training and the right eye in validation), which the report (Section 6.4) names as the other
untested explanation for the validation-to-leaderboard gap.

## Step 1: Find validation images that have a duplicate in the training set (setup)

Loads the per-image hash table from notebook 01 and prepares the 64-bit difference hashes. Flat images (hashes with fewer than 8 or more than 56 set bits) are excluded, exactly as in the duplicate search of report Section 1.4.

In [1]:
# ===== Leakage-controlled validation: re-score saved predictions without the flagged images =====
# Run from notebooks/. No images are read and no model is retrained: it reuses
# ../docs/data_quality_check.csv (md5 + dHash per image) and the saved validation probabilities.
import sys; sys.path.append("../src")  # make the ../src folder importable
import numpy as np, pandas as pd  # numpy: arrays; pandas: tables
import pipeline  # our shared data module (provides the validation table pipeline.val_df)

df = pd.read_csv("../docs/data_quality_check.csv", dtype={"dhash": str})  # per-image table from 01_eda; keep dhash as text
H = np.array([int(h, 16) for h in df["dhash"]], dtype=np.uint64)  # convert each hexadecimal hash string to a 64-bit integer
pc = np.bitwise_count(H)  # number of 1-bits in each hash
idx = np.where((pc >= 8) & (pc <= 56))[0]          # same flat-image filter as the duplicate search
Hs = H[idx]  # hashes of the images that pass the filter

I0000 00:00:1792477076.739915    2789 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792477076.780887    2789 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792477077.594113    2789 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792477078.800786    2789 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7535 MB mem

## Step 2: Search for near-duplicate pairs

Compares every hash with every other hash in blocks of 256 and keeps the pairs that differ in at most 3 of the 64 bits. The result is a list of image pairs (`a`, `b`) that look almost identical.

In [2]:
pairs = []  # will hold the (row, row) index pairs of near-duplicate images
for s in range(0, len(Hs), 256):  # compare 256 hashes at a time to keep memory use low (s = start of the block)
    d = np.bitwise_count(Hs[s:s + 256, None] ^ Hs[None, :])  # XOR the block with all hashes; count differing bits = Hamming distance
    ii, jj = np.nonzero(d <= 3)                      # at most 3 differing bits
    keep = (ii + s) < jj  # keep each pair once and drop an image paired with itself
    pairs.append(np.c_[idx[ii[keep] + s], idx[jj[keep]]])  # convert back to row numbers of df and store as two columns
pairs = np.vstack(pairs); a, b = pairs[:, 0], pairs[:, 1]  # stack all blocks; a and b = first and second image of each pair

## Step 3: Flag the validation images

Keeps only the pairs where one image is in the training subset and the other in the validation subset, and flags the validation-side image. Byte-identical copies (same MD5) are added too. The count printed should be about 143 (report Section 1.4).

In [3]:
split, fn_all = df["split"].values, df["filename"].values  # arrays: train/val label and filename of every image
cross = split[a] != split[b]                         # one image in train, the other in val
val_side = np.where(split[a[cross]] == "val", a[cross], b[cross])  # the validation-side image of each crossing pair
flag = set(fn_all[val_side])  # filenames of validation images with a near-duplicate in training
tr_md5 = set(df.loc[df.split == "train", "md5"])     # exact byte-identical copies as well
flag |= set(df.loc[(df.split == "val") & df.md5.isin(tr_md5), "filename"])  # add validation images that are byte-identical to a training file
print("validation images flagged as duplicate/near-duplicate of a training image:", len(flag))  # CHECK: about 143

validation images flagged as duplicate/near-duplicate of a training image: 143


## Step 4: Re-score the saved predictions with and without the flagged images

For EfficientNetB0, ResNet50V2 and their ensemble, prints the validation errors on all images, on the flagged images only, and on the remaining clean images, with the accuracy on the clean images. If the clean-image accuracy is about the same as the overall accuracy, hash-detected duplicates do not explain the validation-to-leaderboard gap.

In [4]:
y = pipeline.val_df["label"].values  # true validation classes
mask = np.array([f in flag for f in pipeline.val_df["filename"].values])  # True for every flagged validation image
P = {"EfficientNetB0": np.load("../models/val_probs.npy"), "ResNet50V2": np.load("../models/val_probs_r50.npy")}  # saved validation probabilities (run 1)
P["Ensemble (v2)"] = (P["EfficientNetB0"] + P["ResNet50V2"]) / 2  # equal-weight average of the two models
for k, p in P.items():  # evaluate each model in turn
    err = p.argmax(1) != y  # True where the prediction is wrong
    print(f"{k:15s} all: {err.sum():3d} errors ({1-err.mean():.4f}) | flagged {mask.sum()}: {err[mask].sum()} errors | "  # errors on all images, then on the flagged ones
          f"clean {(~mask).sum()}: {err[~mask].sum()} errors, accuracy {1-err[~mask].mean():.4f}")  # errors and accuracy on the clean (unflagged) images

EfficientNetB0  all:  73 errors (0.9943) | flagged 143: 0 errors | clean 12589: 73 errors, accuracy 0.9942
ResNet50V2      all:  63 errors (0.9951) | flagged 143: 0 errors | clean 12589: 63 errors, accuracy 0.9950
Ensemble (v2)   all:  46 errors (0.9964) | flagged 143: 0 errors | clean 12589: 46 errors, accuracy 0.9963
